In [2]:
import json
from urllib.request import urlopen

BASE = "https://cct-ds-code-challenge-input-data.s3.af-south-1.amazonaws.com/"

with urlopen(BASE + "ds_code_challenge_creds.json") as f:
    creds = json.load(f)

for key, value in creds.items():
    if isinstance(value, dict):
        print(key, "->", list(value.keys()))
    else:
        print(key)

s3 -> ['access_key', 'secret_key']


In [3]:
#ask amazon s3 for 3 hexagons and their properties

import boto3

s3 = boto3.client(
    "s3",
    region_name="af-south-1",
    aws_access_key_id=creds["s3"]["access_key"],
    aws_secret_access_key=creds["s3"]["secret_key"],
)

query = "SELECT s.properties FROM S3Object[*].features[*] s LIMIT 3"

response = s3.select_object_content(
    Bucket="cct-ds-code-challenge-input-data",
    Key="city-hex-polygons-8-10.geojson",
    ExpressionType="SQL",
    Expression=query,
    InputSerialization={"JSON": {"Type": "DOCUMENT"}},
    OutputSerialization={"JSON": {}},
)

for event in response["Payload"]:
    if "Records" in event:
        print(event["Records"]["Payload"].decode("utf-8"))

{"properties":{"index":"88ad361801fffff","centroid_lat":-33859427322761434e-15,"centroid_lon":18677843311941835e-15,"resolution":8}}
{"properties":{"index":"88ad361803fffff","centroid_lat":-33855695953283139e-15,"centroid_lon":18668766173796456e-15,"resolution":8}}
{"properties":{"index":"88ad361805fffff","centroid_lat":-33855262542646564e-15,"centroid_lon":18685958700425058e-15,"resolution":8}}



In [4]:
#get entire hexagons including shape and filter to level 8 only
import time

query = """
    SELECT * FROM S3Object[*].features[*] s
    WHERE s.properties.resolution = 8
"""

start = time.perf_counter()

response = s3.select_object_content(
    Bucket="cct-ds-code-challenge-input-data",
    Key="city-hex-polygons-8-10.geojson",
    ExpressionType="SQL",
    Expression=query,
    InputSerialization={"JSON": {"Type": "DOCUMENT"}},
    OutputSerialization={"JSON": {"RecordDelimiter": "\n"}},
)

chunks = []
for event in response["Payload"]:
    if "Records" in event:
        chunks.append(event["Records"]["Payload"])
    elif "Stats" in event:
        stats = event["Stats"]["Details"] #how much Amazon read versus how much it sent

raw = b"".join(chunks).decode("utf-8")
features = [json.loads(line) for line in raw.splitlines() if line] #put each hexagon on its own line

print(len(features), "hexagons")
print(f"took {time.perf_counter() - start:.1f} seconds")
print(f"scanned {stats['BytesScanned'] / 1e6:.0f} MB, returned {stats['BytesReturned'] / 1e6:.1f} MB")

3832 hexagons
took 2.9 seconds
scanned 108 MB, returned 2.0 MB


In [5]:
#check answer from s3 against reference file
import geopandas as gpd

#turn list of hexagons into a table
extracted = gpd.GeoDataFrame.from_features(features, crs="EPSG:4326")
reference = gpd.read_file(BASE + "city-hex-polygons-8.geojson")

print(extracted.columns.tolist())

ours = extracted.set_index("index").sort_index()
ref = reference.set_index("index").sort_index()

print("same count:", len(ours) == len(ref))

missing = set(ref.index) - set(ours.index)
extra = set(ours.index) - set(ref.index)
print("in reference but not ours:", len(missing))
print("in ours but not reference:", len(extra))

same_shape = ours.geometry.geom_equals_exact(ref.geometry, tolerance=1e-9)
print("shapes that match:", same_shape.sum(), "of", len(same_shape))


['geometry', 'index', 'centroid_lat', 'centroid_lon', 'resolution']
same count: True
in reference but not ours: 0
in ours but not reference: 0
shapes that match: 3832 of 3832
